# Introduction

## Support Vector Machine Sentiment Classification

This notebook implements a Support Vector Machine (SVM) model for sentiment classification of Amazon product reviews. The objective is to evaluate the performance of SVM as an alternative classifier for predicting positive and negative sentiments.

Two feature  are tested: 
- TF-IDF features extracted from the processed review text
- A combination of TF-IDF features with additional metadata features including review length, helpful vote count, and verified purchase status. 

The models are evaluated using accuracy, precision, recall, and F1-score.

In [3]:
# Install Neccsary libraries
!pip install joblib pandas scipy scikit-learn

In [20]:
#import neccesary libraries

# save and load trained machine learning models.
import joblib

# handling datasets
import pandas as pd

# Provides scientific computing tools
from scipy.sparse import hstack

import os


# Machine learning library

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

## Load shared Features
These are the files created in your shared preprocessing notebook.

In [4]:
X_train_text = joblib.load("shared/X_train_text.pkl")
X_test_text = joblib.load("shared/X_test_text.pkl")

X_train_meta = joblib.load("shared/X_train_meta.pkl")
X_test_meta = joblib.load("shared/X_test_meta.pkl")

y_train = joblib.load("shared/y_train.pkl")
y_test = joblib.load("shared/y_test.pkl")

 ## Model 1:SVM (TF-IDF Only)

In [5]:
svm_text = LinearSVC()

svm_text.fit(X_train_text, y_train)

pred_text = svm_text.predict(X_test_text)

accuracy_text = accuracy_score(y_test, pred_text)

print("SVM (TF-IDF only)")
print("Accuracy:", accuracy_text)
print(classification_report(y_test, pred_text))

SVM (TF-IDF only)
Accuracy: 0.9271271452743534
              precision    recall  f1-score   support

    negative       0.81      0.64      0.71     23530
    positive       0.94      0.97      0.96    141950

    accuracy                           0.93    165480
   macro avg       0.87      0.81      0.84    165480
weighted avg       0.92      0.93      0.92    165480



The model achieved an accuracy of 92.71% on the test dataset. The classification report shows strong performance in identifying positive reviews (precision = 0.94, recall = 0.97), indicating that the model correctly identifies most positive reviews.

For negative reviews, the model achieved a recall of 0.64 and precision of 0.81, suggesting that while the model detects many negative reviews, some are still misclassified as positive.

This difference in performance is likely influenced by the class imbalance in the dataset, where positive reviews significantly outnumber negative reviews.

In [21]:
svm_text = LinearSVC(class_weight="balanced")

svm_text.fit(X_train_text, y_train)

pred_text = svm_text.predict(X_test_text)

accuracy_text = accuracy_score(y_test, pred_text)

f1_text = f1_score(y_test, pred_text, average="weighted")

print("SVM (TF-IDF only)")
print("Accuracy:", accuracy_text)
print(classification_report(y_test, pred_text))

SVM (TF-IDF only)
Accuracy: 0.8851341551849166
              precision    recall  f1-score   support

    negative       0.56      0.87      0.68     23530
    positive       0.98      0.89      0.93    141950

    accuracy                           0.89    165480
   macro avg       0.77      0.88      0.81    165480
weighted avg       0.92      0.89      0.89    165480




Applying class weighting improved the model’s ability to detect the minority class (negative reviews).  

Before applying class weights, the model showed strong performance for positive reviews but missed many negative reviews. After introducing class weights, the recall for negative reviews increased significantly, meaning the model became better at identifying negative sentiment.  

However, because the dataset remains highly imbalanced, the model still predicts more positive reviews overall. Class weighting therefore helps improve balance in detection performance.


## Model 2: SVM (TF-IDF + Metadata)

In [6]:
# Combine text features with metadata.
X_train_combined = hstack([X_train_text, X_train_meta])
X_test_combined = hstack([X_test_text, X_test_meta])

## Train the Model

In [7]:
svm_combined = LinearSVC()

svm_combined.fit(X_train_combined, y_train)

pred_combined = svm_combined.predict(X_test_combined)

accuracy_combined = accuracy_score(y_test, pred_combined)

print("SVM (TF-IDF + metadata)")
print("Accuracy:", accuracy_combined)
print(classification_report(y_test, pred_combined))

SVM (TF-IDF + metadata)
Accuracy: 0.9271815325114817
              precision    recall  f1-score   support

    negative       0.81      0.64      0.71     23530
    positive       0.94      0.97      0.96    141950

    accuracy                           0.93    165480
   macro avg       0.87      0.81      0.84    165480
weighted avg       0.92      0.93      0.92    165480



The model achieved an accuracy of 92.72%, which is very similar to the performance obtained using TF-IDF features alone. The classification report indicates strong performance in identifying positive reviews (precision = 0.94, recall = 0.97), while performance on negative reviews is lower (recall = 0.64). 
This difference is largely influenced by the class imbalance in the dataset, where positive reviews significantly outnumber negative reviews as stated earlier.

Overall, the addition of metadata features produced only a small improvement in performance, suggesting that the textual information captured through TF-IDF remains the most informative feature for sentiment classification in this dataset.

In [22]:
svm_combined = LinearSVC(class_weight="balanced")

svm_combined.fit(X_train_combined, y_train)

pred_combined = svm_combined.predict(X_test_combined)

accuracy_combined = accuracy_score(y_test, pred_combined)

f1_combined = f1_score(y_test, pred_combined, average="weighted")


print("SVM (TF-IDF + metadata)")
print("Accuracy:", accuracy_combined)
print(classification_report(y_test, pred_combined))

SVM (TF-IDF + metadata)
Accuracy: 0.885321489001692
              precision    recall  f1-score   support

    negative       0.56      0.87      0.68     23530
    positive       0.98      0.89      0.93    141950

    accuracy                           0.89    165480
   macro avg       0.77      0.88      0.81    165480
weighted avg       0.92      0.89      0.89    165480



The model achieved an accuracy of 88.53%. Although accuracy decreased compared to the earlier model without class weighting, the recall for negative reviews improved significantly (0.87). This indicates that the model became more effective at detecting negative reviews, which were previously underrepresented.

Overall, applying class weighting resulted in more balanced detection of both sentiment classes, even though it slightly reduced overall accuracy.

## Save Results



In [23]:
# create folder if it doesn't exist
os.makedirs("models", exist_ok=True)

results = pd.DataFrame({
    "Model": ["SVM", "SVM"],
    "Features": ["TF-IDF", "TF-IDF + Metadata"],
    "Accuracy": [accuracy_text, accuracy_combined],
    "F1 Score": [f1_text, f1_combined]
})

results.to_csv("models/svm_results.csv", index=False)

#save trained models 
joblib.dump(svm_text, "models/svm_tfidf.pkl")
joblib.dump(svm_combined, "models/svm_combined.pkl")

print("Done")

Done


### Code References

The implementation in this notebook relies on several open source libraries and online documentation resources.

- 	Scikit-learn Developers. LinearSVC Documentation. https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html


- OpenAI ChatGPT: Used as a support tool for debugging code, clarifying implementation steps, and interpreting machine learning evaluation results during the development of the notebook.